In [28]:
import pandas as pd
import numpy as np
from tqdm import tqdm

In [29]:
ENTREPOT_PATH = "/home/administrateur/Bureau/Datagrosyst/data_entrepot_outils/"
donnees = {}

def import_df(df_name, path_data, sep, index_col=None):
    donnees[df_name] = pd.read_csv(path_data+df_name+'.csv', sep = sep, index_col=index_col, low_memory=False).replace({'\r\n': '\n'}, regex=True)

def import_dfs(df_names, path_data, sep = ',', index_col=None, verbose=False):
    for df_name in tqdm(df_names) : 
        if(verbose) :
            print(" - ", df_name)
        import_df(df_name, path_data, sep, index_col=index_col)

tables_with_id = [
    'composant_culture',
    'culture',
    'espece',
    'recolte_rendement_prix',
    'recolte_rendement_prix_restructure'
]

tables_without_id = [
]


# import des données de l'entrepôt avec la colonne 'id' en index 
import_dfs(tables_with_id, ENTREPOT_PATH, sep = ',', index_col='id', verbose=False)

# import des données du magasin
import_dfs(tables_without_id, ENTREPOT_PATH, sep = ',', verbose=False)

100%|██████████| 5/5 [00:08<00:00,  1.68s/it]
0it [00:00, ?it/s]


In [30]:
culture = donnees['culture'].reset_index().rename(columns={'id':'culture_id'}).copy()
cc = donnees['composant_culture'].reset_index().rename(columns={'id':'composant_culture_id'}).copy()
# espece = donnees['espece'].rename(columns={'id':'espece_id'}).reset_index().copy()


espece = pd.read_csv('/home/administrateur/Bureau/' + 'MAE_ref_espece_gcpe.csv').rename(columns={'id':'espece_id'})

MAE_matrice_precise = pd.read_csv('/home/administrateur/Bureau/' + 'MAE_matrice_passage_dirodur_espece_precise.csv').rename(columns={'id':'espece_id'})

MAE_matrice_famille = pd.read_csv('/home/administrateur/Bureau/' + 'MAE_matrice_passage_dirodur_famille_bota.csv').rename(columns={'id':'espece_id'})


with open('/home/administrateur/Bureau/' + 'liste_culture_id_gcpe.txt', "r") as f:
    list_cid_gcpe = [line.strip() for line in f.readlines()]

In [31]:
df = cc.merge(espece, how='left', on='espece_id')
# On ne garde que les culture_id qui ont été utilisées dans un sdc GCPE
culture = culture.loc[culture['culture_id'].isin(list_cid_gcpe)]
df = df.merge(culture, how='right', on='culture_id')

In [32]:
def concat_unique_sorted(series):
    cleaned = series.dropna().unique()
    if len(cleaned) == 0:
        return np.nan
    return '_'.join(sorted(cleaned))
def get_nb_unique_typo(series):
    cleaned = series.dropna().unique()
    return len(cleaned)

In [33]:
agg_dict = {
    'nb_composant_culture': 'sum',
    'typodirodur_espece_precise': concat_unique_sorted,
    'typodirodur_espece_famille_bota': concat_unique_sorted,
    'typodirodur_espece_periode_semis': concat_unique_sorted,
    'nb_typodirodur_espece_precise': get_nb_unique_typo,
    'nb_typodirodur_espece_famille_bota': get_nb_unique_typo,
}

In [34]:
len(df.loc[df['composant_culture_id'].isna(), 'nom'].unique().tolist())

195

In [35]:
df['nb_composant_culture'] = 1
df['nb_typodirodur_espece_precise'] = df['typodirodur_espece_precise']
df['nb_typodirodur_espece_famille_bota'] = df['typodirodur_espece_famille_bota']

df2 = df[['culture_id',
         'nb_composant_culture',
         'typodirodur_espece_precise',
         'typodirodur_espece_famille_bota',
         'typodirodur_espece_periode_semis',
         'nb_typodirodur_espece_precise',
         'nb_typodirodur_espece_famille_bota']].groupby('culture_id').agg(agg_dict).reset_index()

In [36]:
df2.typodirodur_espece_precise.value_counts(dropna=False).sort_values(ascending=False)

typodirodur_espece_precise
Blé tendre Hiver Meunier                                                                          9226
Maïs                                                                                              8734
Blé tendre Hiver Biscuit                                                                          6696
Colza Oléagineux Hiver                                                                            6653
Orge 6 rangs Hiver                                                                                4666
                                                                                                  ... 
Fléole des prés_Fétuque des prés_Fétuque rouge_Pâturin des prés_Ray-grass anglais_Trèfle blanc       1
Fléole des prés_Fétuque rouge_Pâturin des prés_Ray-grass anglais_Trèfle blanc_Trèfle violet          1
Fléole bulbeuse_Fétuque élevée                                                                       1
Alpiste roseau_Dactyle_Fétuque élevée_Luzerne_

In [37]:
df2.typodirodur_espece_famille_bota.value_counts(dropna=False).sort_values(ascending=False)

typodirodur_espece_famille_bota
Poaceae                                                       48765
Fabaceae                                                      11479
Brassicaceae                                                   7127
Fabaceae_Poaceae                                               6826
NaN                                                            5383
                                                              ...  
Asteraceae_Fabaceae_Poaceae_Polygonaceae                          1
Asteraceae_Fabaceae_Polygonaceae                                  1
Amaranthaceae_Cucurbitaceae_Fabaceae                              1
Brassicaceae_Fabaceae_Hydrophyllaceae_Poaceae_Polygonaceae        1
Saxifragaceae                                                     1
Name: count, Length: 96, dtype: int64

In [38]:
df3 = df2.loc[df2['nb_typodirodur_espece_precise'] != 1]
df3 = df3.groupby(['typodirodur_espece_precise','typodirodur_espece_periode_semis']).size().sort_values(ascending=False).reset_index()
df3.rename(columns={0:'count'}, inplace=True)
# df3.to_csv('/home/administrateur/Bureau/' + "matrice_passage_dirodur_espece_precise.csv", index=False)
df3

,typodirodur_espece_precise,typodirodur_espece_periode_semis,count
0,Pois Fourrager / Fourrage Hiver_Triticale,hiver,279
1,Fétuque élevée_Ray-grass anglais_Trèfle blanc,pluriannuelle,245
2,Blé tendre Hiver Amidon_Blé tendre Hiver Meunier,hiver,227
3,Blé tendre Hiver Meunier_Fèverole Hiver,hiver,200
4,Pois Protéagineux Hiver_Triticale,hiver,175
...,...,...,...
1784,Alpiste roseau_Fétuque élevée_Pâturin des prés...,pluriannuelle,1
1785,Colza Oléagineux Hiver_Fèverole Printemps Grai...,automne_ete_pluriannuelle_printemps,1
1786,Dactyle_Fléole des prés_Fétuque des prés_Fétuq...,pluriannuelle,1
1787,Dactyle_Fléole des prés_Fétuque des prés_Lotie...,pluriannuelle,1


In [ ]:
df4 = df2.loc[df2['nb_typodirodur_espece_famille_bota'] != 1]
df4 = df4.groupby(['typodirodur_espece_famille_bota','typodirodur_espece_periode_semis']).size().sort_values(ascending=False).reset_index()
df4.rename(columns={0:'count'}, inplace=True)
df4

,typodirodur_espece_famille_bota,typodirodur_espece_periode_semis,count
0,Fabaceae_Poaceae,hiver,3133
1,Fabaceae_Poaceae,pluriannuelle,2320
2,Fabaceae_Poaceae,hiver_printemps,294
3,Fabaceae_Poaceae,hiver_pluriannuelle,223
4,Fabaceae_Poaceae,printemps,155
...,...,...,...
172,Plantaginaceae_Fabaceae_Poaceae,hiver_pluriannuelle,1
173,Brassicaceae_Fabaceae_Poaceae,automne_pluriannuelle_printemps,1
174,Brassicaceae_Fabaceae_Poaceae,ete,1
175,Brassicaceae_Fabaceae_Poaceae,hiver_pluriannuelle,1


In [ ]:
df3 = df3.merge(MAE_matrice_precise[['typodirodur_espece_precise', 
                     'typodirodur_culture',
                     'culture_avec_compagne', 
                     'culture_annuelle_asso', 
                     'prairie']], on='typodirodur_espece_precise', how='left')
df3

,typodirodur_espece_precise,typodirodur_espece_periode_semis,count,typodirodur_culture,culture_avec_compagne,culture_annuelle_asso,prairie
0,Pois Fourrager / Fourrage Hiver_Triticale,hiver,279,asso_cer_leg,non,oui,non
1,Fétuque élevée_Ray-grass anglais_Trèfle blanc,pluriannuelle,245,PT_cer_leg,non,non,oui
2,Blé tendre Hiver Amidon_Blé tendre Hiver Meunier,hiver,227,Blé tendre,non,non,non
3,Blé tendre Hiver Meunier_Fèverole Hiver,hiver,200,asso_cer_leg,non,oui,non
4,Pois Protéagineux Hiver_Triticale,hiver,175,asso_cer_leg,non,oui,non
...,...,...,...,...,...,...,...
1784,Alpiste roseau_Fétuque élevée_Pâturin des prés...,pluriannuelle,1,PT_cer_leg,non,non,oui
1785,Colza Oléagineux Hiver_Fèverole Printemps Grai...,automne_ete_pluriannuelle_printemps,1,Colza,oui,oui,non
1786,Dactyle_Fléole des prés_Fétuque des prés_Fétuq...,pluriannuelle,1,PT_cer_leg,non,non,oui
1787,Dactyle_Fléole des prés_Fétuque des prés_Lotie...,pluriannuelle,1,PT_cer_leg,non,non,oui


In [ ]:
# df4 = df4.merge(MAE_matrice_precise[['typodirodur_espece_famille_bota', 
#                      'typodirodur_culture',
#                      'culture_avec_compagne', 
#                      'culture_annuelle_asso', 
#                      'prairie']], on='typodirodur_espece_famille_bota', how='left')

df4['duplic_date_semis'] = np.nan
df4.loc[df4.duplicated(subset=['typodirodur_espece_famille_bota'], keep=False), 'duplic_semis'] = 'check_date_semis'

# df4.to_csv('/home/administrateur/Bureau/' + "matrice_passage_dirodur_famille_bota.csv", index=False)
df4